In [ ]:
import os
import json
import warnings
import numpy as np
import xarray as xr
import proplot as pplt
from matplotlib.lines import Line2D
warnings.filterwarnings('ignore')
pplt.rc.update({
    'savefig.dpi':900,
    'savefig.bbox':'tight',
    'savefig.pad_inches':0.02,
    'tick.minor':False,
    'font.size':9,
    'label.size':9,
    'tick.labelsize':9,
    'title.size':9,
    'abc.size':9,
    'legend.fontsize':9,
    'suptitle.size':9,
    'leftlabelsize':9,
    'toplabelsize':9,
    'leftlabel.weight':'normal',
    'toplabel.weight':'normal',
    'reso':'xx-hi'})

In [ ]:
with open('../scripts/configs.json','r',encoding='utf-8') as f:
    CONFIGS = json.load(f)
SPLITSDIR  = CONFIGS['filepaths']['splits']
WEIGHTSDIR = CONFIGS['filepaths']['weights']
PREDSDIR   = CONFIGS['filepaths']['predictions']
FIELDVARS  = CONFIGS['experiments']['sr']['runs']['sr_atm']['fieldvars']
SEEDS      = CONFIGS['experiments']['nn']['seeds']
TARGETVAR  = CONFIGS['domain']['target']
STATSFILE  = os.path.join(SPLITSDIR,'stats.json')
with open(STATSFILE,'r',encoding='utf-8') as f:
    STATS = json.load(f)
TMEAN = STATS[f'{TARGETVAR}_mean']
TSTD  = STATS[f'{TARGETVAR}_std']
ZMIN  = (0.0-TMEAN)/TSTD

# 1. Upper Bound (zmax)

Load the target from train+valid splits and examine its distribution in z-space and mm-space to find a reasonable prediction ceiling.

In [ ]:
zvals = []
for split in ['train','valid']:
    with xr.open_dataset(os.path.join(SPLITSDIR,f'{split}.h5'),engine='h5netcdf') as ds:
        target = ds[TARGETVAR].transpose('time','lat','lon').values.ravel()
        zvals.append(target[np.isfinite(target)])
zall = np.concatenate(zvals)
print(f'Total samples: {len(zall):,}')
print(f'z-scored target stats: mean={zall.mean():.4f}  std={zall.std():.4f}  min={zall.min():.4f}  max={zall.max():.4f}')
print(f'ZMIN (z-score of 0 mm): {ZMIN:.4f}')
print()
for p in [99,99.5,99.9,99.95,99.99,99.999,100]:
    zp = np.percentile(zall,p)
    mmval = np.expm1(zp*TSTD+TMEAN)
    print(f'  p{p:>7}: z={zp:7.3f}  →  {mmval:10.2f} mm')

In [ ]:
fig,axs = pplt.subplots(ncols=2,refwidth=3,refheight=2.5)
axs.format(grid=False)

axs[0].hist(zall,bins=200,color='gray5',alpha=0.6)
axs[0].axvline(ZMIN,color='C3',linewidth=1.5,linestyle='--',label=f'zmin={ZMIN:.2f}')
for p,ls in [(99.9,'--'),(99.99,'-.'),(99.999,':')]:
    zp = np.percentile(zall,p)
    axs[0].axvline(zp,color='C0',linewidth=1,linestyle=ls,label=f'p{p}={zp:.2f}')
axs[0].format(xlabel='z-scored log1p(tp)',ylabel='Count',title='Target Distribution (z-space)',yscale='log')
axs[0].legend(loc='ur')

mmall = np.expm1(zall*TSTD+TMEAN)
axs[1].hist(mmall[mmall>0.01],bins=np.logspace(-2,2.5,200),color='gray5',alpha=0.6)
for p,ls in [(99.9,'--'),(99.99,'-.'),(99.999,':')]:
    mmval = np.percentile(mmall,p)
    axs[1].axvline(mmval,color='C0',linewidth=1,linestyle=ls,label=f'p{p}={mmval:.1f} mm')
axs[1].format(xlabel='Total Precipitation (mm)',ylabel='Count',title='Target Distribution (mm-space)',xscale='log',yscale='log')
axs[1].legend(loc='ur')

fig.format(suptitle='Upper Bound Analysis')
pplt.show()

# 2. Monotonicity Cube Test (Grundner et al. 2024)

For each atmospheric predictor (RH, $\theta_e$, $\theta_e^*$), hold the other two approximately constant by binning them into $N^2$ cubes, then fit a linear slope in each cube. Report the fraction of cubes where the expected monotonicity sign holds.

Expected signs:
- $\partial P / \partial \mathrm{RH} \geq 0$
- $\partial P / \partial \theta_e \geq 0$
- $\partial P / \partial \theta_e^* \leq 0$

In [ ]:
with xr.open_dataset(os.path.join(SPLITSDIR,'train.h5'),engine='h5netcdf') as ds:
    ntime,nlat,nlon = ds.time.size,ds.lat.size,ds.lon.size
    nsig = ds.sizes.get('sig',1)
    dsig = ds.dsig.values
    fields = np.stack([ds[v].transpose('time','lat','lon','sig').values.reshape(-1,nsig) for v in FIELDVARS],axis=1)
    surfmask = ds.surfmask.transpose('time','lat','lon','sig').values.reshape(-1,nsig) if 'surfmask' in ds else None
    flat = lambda v: ds[v].transpose('time','lat','lon').values.ravel() if 'time' in ds[v].dims else np.tile(ds[v].values,(ntime,1,1)).ravel()
    obstrain = flat('tp')
    lftrain = flat('lf')

kernels = []
for seed in SEEDS:
    with xr.open_dataset(os.path.join(WEIGHTSDIR,f'nn_gauss_{seed}_weights.nc'),engine='h5netcdf') as ds:
        kernels.append(ds.k.values)

weights = fields*np.mean(kernels,axis=0)[None,:,:]*dsig[None,None,:]
integrals = (weights*surfmask[:,None,:] if surfmask is not None else weights).sum(axis=2)
RHTRAIN,TETRAIN,TESTRAIN = integrals[:,0],integrals[:,1],integrals[:,2]

validmask = np.isfinite(obstrain)&np.isfinite(RHTRAIN)&np.isfinite(TETRAIN)&np.isfinite(TESTRAIN)
OBSTRAIN = obstrain[validmask]
RHTRAIN  = RHTRAIN[validmask]
TETRAIN  = TETRAIN[validmask]
TESTRAIN = TESTRAIN[validmask]
LFTRAIN  = lftrain[validmask]
print(f'Valid training samples: {validmask.sum():,}')

In [ ]:
def cube_monotonicity_test(target,separated,controlA,controlB,expected_sign,
                           ncubes_list=[3,4,5,6,7],minsamples=10000,plo=1,phi=99):
    results = []
    for N in ncubes_list:
        edgesA = np.linspace(*np.percentile(controlA,[plo,phi]),N+1)
        edgesB = np.linspace(*np.percentile(controlB,[plo,phi]),N+1)
        nsatisfied = 0
        ntested = 0
        slopes = []
        for i in range(N):
            for j in range(N):
                mask = ((controlA>=edgesA[i])&(controlA<edgesA[i+1])&
                        (controlB>=edgesB[j])&(controlB<edgesB[j+1]))
                if mask.sum()<minsamples:
                    continue
                xsub = separated[mask]
                ysub = target[mask]
                xmean = xsub.mean()
                slope = np.sum((xsub-xmean)*(ysub-ysub.mean()))/np.sum((xsub-xmean)**2)
                ntested += 1
                slopes.append(slope)
                if expected_sign>=0 and slope>=0:
                    nsatisfied += 1
                elif expected_sign<0 and slope<=0:
                    nsatisfied += 1
        pct = 100*nsatisfied/ntested if ntested>0 else np.nan
        results.append(dict(N=N,ncubes=N**2,ntested=ntested,nsatisfied=nsatisfied,
                            pct=pct,slopes=slopes))
    return results

CONSTRAINTS = {
    'RH':       dict(sep=RHTRAIN, ctrlA=TETRAIN,  ctrlB=TESTRAIN, sign=+1),
    'thetae':   dict(sep=TETRAIN, ctrlA=RHTRAIN,  ctrlB=TESTRAIN, sign=+1),
    'thetaestar':dict(sep=TESTRAIN,ctrlA=RHTRAIN,  ctrlB=TETRAIN,  sign=-1)}

CUBERESULTS = {}
for name,spec in CONSTRAINTS.items():
    sign_label = '≥ 0' if spec['sign']>=0 else '≤ 0'
    print(f'\nPC: ∂P/∂{name} {sign_label}')
    res = cube_monotonicity_test(OBSTRAIN,spec['sep'],spec['ctrlA'],spec['ctrlB'],spec['sign'])
    CUBERESULTS[name] = res
    for r in res:
        print(f"  N={r['N']:2d}  cubes={r['ncubes']:3d}  tested={r['ntested']:3d}  "
              f"satisfied={r['nsatisfied']:3d}  ({r['pct']:.0f}%)")

In [ ]:
fig,axs = pplt.subplots(ncols=3,refwidth=2.5,refheight=2)
axs.format(grid=False,ylabel='% Cubes Satisfied',ylim=(0,105),xlabel='N (cubes per control axis)')

for i,(name,res) in enumerate(CUBERESULTS.items()):
    sign_label = '≥ 0' if CONSTRAINTS[name]['sign']>=0 else '≤ 0'
    ns = [r['N'] for r in res]
    pcts = [r['pct'] for r in res]
    axs[i].plot(ns,pcts,color='k',marker='o',linewidth=1.5)
    axs[i].axhline(75,color='C3',linewidth=0.8,linestyle='--',label='75% threshold')
    axs[i].format(title=f'∂P/∂{name} {sign_label}',xticks=ns)
    axs[i].legend(loc='lr')

fig.format(suptitle='Monotonicity Cube Test (Grundner et al. 2024)')
pplt.show()

# 3. Cube Test by Region (Land vs Ocean)

Repeat the cube test separately for land and ocean to check whether constraints differ by surface type.

In [ ]:
REGIONMASKS = {'Land':LFTRAIN>0.5,'Ocean':LFTRAIN<0.5}

for region,rmask in REGIONMASKS.items():
    print(f'\n=== {region} ({rmask.sum():,} samples) ===')
    for name,spec in CONSTRAINTS.items():
        sign_label = '≥ 0' if spec['sign']>=0 else '≤ 0'
        res = cube_monotonicity_test(
            OBSTRAIN[rmask],spec['sep'][rmask],spec['ctrlA'][rmask],spec['ctrlB'][rmask],
            spec['sign'],ncubes_list=[4,5,6])
        best = res[-1]
        print(f'  ∂P/∂{name:12s} {sign_label}:  {best["pct"]:.0f}% at N={best["N"]} '
              f'({best["nsatisfied"]}/{best["ntested"]} cubes)')

# 4. Test Current SR Equations Against Constraints

Evaluate monotonicity of each optimized SR equation via finite differences on unstandardized predictors. Report violation fraction across the training distribution.

In [ ]:
import pandas as pd
import ast as _ast
import re

SRFUNCTIONS = {
    'cube':lambda x:x**3,'square':lambda x:x**2,'neg':lambda x:-x,
    'sqrt':np.sqrt,'exp':np.exp,'log':np.log,'abs':np.abs,
    'sin':np.sin,'cos':np.cos,'max':np.maximum,'min':np.minimum,
    '_safepow':lambda a,b:np.abs(a)**b}

def _prepare_form(form):
    return re.sub(r'(\w+)\^(\w+)',r'_safepow(\1,\2)',form)

def expand_form(form,registry):
    for eqname,entry in registry.iterrows():
        if entry['name'] in form:
            form = form.replace(entry['name'],f'({entry["form"]})')
    return form

def extract_constants(form,predictornames):
    names = {node.id for node in _ast.walk(_ast.parse(form,mode='eval'))
             if isinstance(node,_ast.Name)}
    return sorted(names-set(predictornames)-set(SRFUNCTIONS)-{'True','False','None'})

def eval_form(form,x,predictornames,constants):
    ns = dict(SRFUNCTIONS,__builtins__={})
    for pname in predictornames:
        ns[pname] = x[pname].values
    ns.update(constants)
    out = eval(_prepare_form(form),ns)
    if np.ndim(out)==0:
        out = np.full(len(x),float(out))
    return np.asarray(out,dtype=float)

OPTIMIZED = pd.read_csv('../models/sr/optimized_equations.csv')
OPTIMIZED

In [ ]:
PREDICTORDATA = {
    'rh':RHTRAIN,'thetae':TETRAIN,'thetaestar':TESTRAIN,
    'lf':LFTRAIN,'shf':np.zeros_like(RHTRAIN),'lhf':np.zeros_like(RHTRAIN)}

with xr.open_dataset(os.path.join(SPLITSDIR,'train.h5'),engine='h5netcdf') as ds:
    flat = lambda v: ds[v].transpose('time','lat','lon').values.ravel() if 'time' in ds[v].dims else np.tile(ds[v].values,(ntime,1,1)).ravel()
    shftrain,lhftrain = flat('shf'),flat('lhf')
PREDICTORDATA['shf'] = shftrain[validmask]
PREDICTORDATA['lhf'] = lhftrain[validmask]

ATMCONSTRAINTS = {
    'rh':       +1,
    'thetae':   +1,
    'thetaestar':-1}

EPS = 1e-3

for _,row in OPTIMIZED.iterrows():
    name = row['name']
    form = row['form']
    constants = json.loads(row['constants'])
    if name=='sr_bl_eq':
        continue
    expanded = expand_form(form,OPTIMIZED)
    for _,baserow in OPTIMIZED.iterrows():
        if baserow['name'] in form and baserow['name']!=name:
            baseconstants = json.loads(baserow['constants'])
            for k,v in baseconstants.items():
                if k not in constants:
                    constants[k] = v
    allpredictors = sorted(set(extract_constants(expanded,list(PREDICTORDATA.keys())))
                           .symmetric_difference(set()) | set(),
                           key=lambda x:x)
    predictornames = [p for p in PREDICTORDATA if p in expanded]
    x = pd.DataFrame({p:PREDICTORDATA[p] for p in predictornames})
    raw = eval_form(expanded,x,predictornames,constants)
    pred = ZMIN+np.maximum(raw,0.0)
    active = raw>0
    print(f'\n{name}: {form}')
    if expanded!=form:
        print(f'  Expanded: {expanded}')
    print(f'  Constants: {constants}')
    print(f'  Active region (raw>0): {active.mean()*100:.1f}%')
    for var,expected_sign in ATMCONSTRAINTS.items():
        if var not in predictornames:
            continue
        xplus = x.copy()
        xplus[var] = xplus[var]+EPS
        rawplus = eval_form(expanded,xplus,predictornames,constants)
        dpred = (rawplus-raw)/EPS
        sign_label = '≥ 0' if expected_sign>=0 else '≤ 0'
        if expected_sign>=0:
            violations = dpred<0
        else:
            violations = dpred>0
        violations_active = violations&active
        pct_all = violations.mean()*100
        pct_active = violations_active.sum()/active.sum()*100 if active.any() else 0
        print(f'  ∂P/∂{var:12s} {sign_label}:  violated in {pct_all:.1f}% of all samples, '
              f'{pct_active:.1f}% of active region')

# 5. What zmax Would Clip

For candidate zmax values, show how many training/test predictions from each model would be clipped, and the effect on R².

In [ ]:
SPLIT = 'test'
ALLMODELS = {**CONFIGS['experiments']['nn']['runs'],**CONFIGS['experiments']['sr']['optimizedeqs']}
ORDER = ['nn_gauss']+[name for name in CONFIGS['experiments']['sr']['optimizedeqs']
          if os.path.exists(os.path.join(PREDSDIR,f'{name}_{SPLIT}_predictions.nc'))]
LABELS = {n:ALLMODELS[n]['description'] for n in ORDER}

with xr.open_dataset(os.path.join(SPLITSDIR,f'{SPLIT}.h5'),engine='h5netcdf') as ds:
    obstest = ds[TARGETVAR].transpose('time','lat','lon').values.ravel()

predtest = {}
for name in ORDER:
    with xr.open_dataset(os.path.join(PREDSDIR,f'{name}_{SPLIT}_predictions.nc')) as ds:
        pred = ds[TARGETVAR].load()
    if 'seed' in pred.dims: pred = pred.mean('seed')
    if 'complexity' in pred.dims: pred = pred.isel(complexity=0)
    predtest[name] = pred.values.ravel()

def get_r2(obs,pred):
    mask = np.isfinite(obs)&np.isfinite(pred)
    o,p = obs[mask],pred[mask]
    return 1-np.sum((o-p)**2)/np.sum((o-o.mean())**2)

zmax_candidates = np.percentile(zall,[99,99.5,99.9,99.95,99.99,99.999])
pct_labels = ['p99','p99.5','p99.9','p99.95','p99.99','p99.999']

print(f'{"zmax":>10s} {"mm equiv":>10s}',end='')
for name in ORDER:
    print(f' {LABELS[name]:>12s}',end='')
print()
print(f'{"":>10s} {"":>10s}',end='')
for name in ORDER:
    r2orig = get_r2(obstest,predtest[name])
    print(f' {r2orig:12.4f}',end='')
print('  (no clip)')

for zmax,plabel in zip(zmax_candidates,pct_labels):
    mmmax = np.expm1(zmax*TSTD+TMEAN)
    print(f'{plabel:>10s} {mmmax:10.1f}',end='')
    for name in ORDER:
        clipped = np.minimum(predtest[name],mmmax)
        r2clip = get_r2(obstest,clipped)
        print(f' {r2clip:12.4f}',end='')
    print()